In [89]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [90]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [91]:
# Function to generate a new dataset
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [92]:
# Generate the dataset and write it to 'dataset.json'
dataset = generate_dataset()
with open("dataset-code-model.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [93]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [94]:
# import textwrap

# Passes a test case into Claude
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    # return textwrap.dedent(output).lstrip()
    return output

In [95]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


In [96]:
# Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [97]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [98]:
with open("dataset-code-model.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 8.166666666666666


In [99]:
# print(json.dumps(results, indent=2))
# print(results)
for i, item in enumerate(results, 1):
    print(f"\n=== Test Case {i} ===")
    print("Task:", item["test_case"]["task"])
    print("\nOutput:\n")
    print(item["output"])   # ← THIS is the key
    print("\nScore:", item["score"])
    print("\nReasoning:", item["reasoning"])
    print("-" * 60)

with open("evaluation.json", "w") as f:
    json.dump(results, f, indent=2)


=== Test Case 1 ===
Task: Parse an AWS S3 bucket name from an S3 URI string like 's3://my-bucket-name/path/to/object'

Output:


import re

def parse_s3_bucket_name(s3_uri):
    match = re.match(r's3://([a-zA-Z0-9._-]+)/', s3_uri)
    if match:
        return match.group(1)
    return None


Score: 8.0

Reasoning: The solution works for the common case of 's3://bucket-name/path/to/object' but fails for valid S3 URIs without a trailing slash. The regex pattern should use a non-capturing group or optional trailing slash. Additionally, the implementation lacks input validation and doesn't enforce AWS bucket naming constraints, which could mask invalid inputs.
------------------------------------------------------------

=== Test Case 2 ===
Task: Create a CloudFormation template JSON object that defines a basic S3 bucket with versioning enabled

Output:


{
  "AWSTemplateFormatVersion": "2010-09-09",
  "Description": "CloudFormation template that defines a basic S3 bucket with versioning 